In [101]:
import os
from dotenv import load_dotenv
from roboflow import Roboflow

load_dotenv()
api_key = os.environ.get("ROBOFLOW_API_KEY")

rf = Roboflow(api_key=api_key)
project = rf.workspace("cybertech-qde01").project("waste-classification-q75av-awlnx")
version = project.version(1)
dataset = version.download("multiclass")

loading Roboflow workspace...
loading Roboflow project...
loading Roboflow project...


In [102]:
import os
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base_path = dataset.location

class_names = ["cardboard", "glass", "metal", "paper", "plastic"]

def load_roboflow_split(split_name, batch_size=32, image_size=(224, 224)):
    folder_path = os.path.join(base_path, split_name)
    csv_path = os.path.join(folder_path, "_classes.csv")
        
    df = pd.read_csv(csv_path)

    # Filter trash from training
    if split_name == "train":
        df = df[df[class_names].sum(axis=1) > 0]
        
        # Rotate and change training images
        datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=30,
            width_shift_range=0.2,
            height_shift_range=0.2,
            shear_range=0.1,
            zoom_range=0.2,
            horizontal_flip=True,
            fill_mode='nearest'
        )
    else:
        # For validating and testing just rescale
        datagen = ImageDataGenerator(rescale=1./255)
    
    generator = datagen.flow_from_dataframe(
        dataframe=df,
        directory=folder_path,
        x_col="filename",
        y_col=class_names, 
        target_size=image_size,
        batch_size=batch_size,
        class_mode="raw", 
        shuffle=(split_name == "train") 
    )
    
    return generator

train_gen = load_roboflow_split("train", batch_size=64)
valid_gen = load_roboflow_split("valid", batch_size=64)
test_gen = load_roboflow_split("test", batch_size=64)

Found 1674 validated image filenames.
Found 504 validated image filenames.
Found 253 validated image filenames.
Found 504 validated image filenames.
Found 253 validated image filenames.


In [103]:
import matplotlib.pyplot as plt
import numpy as np

def plot_image(images, labels=None, predictions=None, cmap=None, n_preds=10):
    show_label = True
    if labels is None:
        labels = images
        show_label = False

    show_prediction = True
    if predictions is None:
        predictions = images
        show_prediction = False

    n_preds = min(n_preds, len(images))

    for image, label, prediction in zip(images[:n_preds], labels[:n_preds], predictions[:n_preds]):
        plt.xticks([])
        plt.yticks([])
        plt.imshow(image, cmap=cmap)

        label_text = ""

        if show_prediction:
          cls_pred = tf.math.argmax(prediction, axis=0)
          class_number = class_names[cls_pred] 

          label_text = f"\npredicted: {class_names[cls_pred]}, probability: {round(prediction[cls_pred], 2)}"

        if show_label:
            class_idx = np.argmax(label)
            plt.xlabel(f"{class_names[class_idx]}" + label_text)

        plt.show()

    if show_prediction:
      matches = tf.keras.metrics.categorical_accuracy(labels[:n_preds], predictions[:n_preds])
      print("accuracy on this plotted sample: ", tf.reduce_mean(matches).numpy())

In [104]:

backbone = tf.keras.applications.ResNet50V2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
backbone.trainable = False


model = tf.keras.Sequential([
    # input = generator's output size (224, 224, 3)
    tf.keras.Input(shape=(224, 224, 3)),

    #backbone,

    # Replacing backbone with code below gives us ~66 f1 score:

    #Conv2D - Slides the kernel
    #First arg (32) is the amount of filters to learn (the numbers in the kernel)
    #Early layers should have fewer filters because they learn simpler shapes (eg. vertical lines)
    #tf.keras.layers.Conv2D(32, (3,3), activation='relu'),

    #MaxPooling2D - Shrinks image size by picking max number in a given pool ("window"), removing noise
    #default pool size is (2,2) which means it's shrinking the image by half in width and height
    #tf.keras.layers.MaxPooling2D(), 
    
    #Learning medium complexity shapes
    #tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    #tf.keras.layers.MaxPooling2D(),
    
    #Learning advanced shapes
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(256, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),


    tf.keras.layers.Dense(5, activation="softmax")
])

model.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_26 (Conv2D)              │ (None, 222, 222, 128)  │         3,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_63 (MaxPooling2D) │ (None, 111, 111, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_27 (Conv2D)              │ (None, 109, 109, 256)  │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_64 (MaxPooling2D) │ (None, 54, 54, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_10     │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 332,293 (1.27 MB)

 Trainable params: 332,293 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [105]:
loss = tf.keras.losses.CategoricalCrossentropy()

metrics = [
    tf.keras.metrics.CategoricalAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='auc')
]

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=10, 
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='best_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.2, 
        patience=5, 
        min_lr=1e-6,
        verbose=1
    )
]

model.compile(
    loss=loss,
    optimizer='adamw',
    metrics=metrics
)

history = model.fit(
    train_gen, 
    epochs=50, 
    validation_data=valid_gen,
    callbacks=callbacks
)

Epoch 1/50
 1/27 ━━━━━━━━━━━━━━━━━━━━ 1:54 4s/step - accuracy: 0.1875 - auc: 0.5135 - loss: 1.6123 - precision: 0.0000e+00 - recall: 0.0000e+00

KeyboardInterrupt: 

In [ ]:
def show_batch_preds():
    X_test_batch, y_test_batch = next(iter(test_gen))
    test_preds = model.predict(X_test_batch)
    plot_image(X_test_batch, y_test_batch, test_preds)

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

def extract_average_f1(report, extra_f1=[]):
    f1_scores = []
    f1_scores.extend(extra_f1)
    for cls in class_names:
        if cls in report:
            f1_scores.append(report[cls]['f1-score'])
            
    return np.mean(f1_scores) if f1_scores else 0

def predict_from_generator(generator):
    generator.reset()
    
    probabilities = model.predict(generator)
    predicted_classes = np.argmax(probabilities, axis=1)
    
    y_true_vectors = generator.labels
    
    y_true = np.argmax(y_true_vectors, axis=1)
    
    report_string = classification_report(y_true, predicted_classes, target_names=class_names, zero_division=0)
    print(report_string)
    
    return classification_report(y_true, predicted_classes, target_names=class_names, output_dict=True, zero_division=0)

print("\n--- TEST REPORT IGNORING TRASH ---:")
report_test = predict_from_generator(test_gen)

print("\n--- VALIDATION REPORT IGNORING TRASH ---:")
report_valid = predict_from_generator(valid_gen)

# We throw in an extra 0 during calculation since in this sample trash always has an f1 score of 0
avg_f1 = np.mean([extract_average_f1(report_test, [0]), extract_average_f1(report_valid, [0])])
print(f"Average f1 across test/valid: {avg_f1:.4f}")


--- TEST REPORT IGNORING TRASH ---:
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 152ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 152ms/step
              precision    recall  f1-score   support

   cardboard       0.78      0.60      0.68        58
       glass       0.45      0.40      0.43        47
       metal       0.42      0.45      0.43        40
       paper       0.64      0.89      0.75        64
     plastic       0.74      0.57      0.64        44

    accuracy                           0.61       253
   macro avg       0.60      0.58      0.59       253
weighted avg       0.62      0.61      0.60       253


--- VALIDATION REPORT IGNORING TRASH ---:
              precision    recall  f1-score   support

   cardboard       0.78      0.60      0.68        58
       glass       0.45      0.40      0.43        47
       metal       0.42      0.45      0.43        40
       paper       0.64      0.89      0.75        64
     plastic       0.74      0.57      0.64        44

    accuracy               

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

def predict_with_threshold(generator, threshold=0.5, print_report=True):
    generator.reset()
    
    raw_probs = model.predict(generator)
    pred_classes = np.argmax(raw_probs, axis=1)
    confidence = np.max(raw_probs, axis=1)
    
    TRASH_INDEX = 5
    low_confidence_mask = confidence < threshold
    pred_classes[low_confidence_mask] = TRASH_INDEX
    
    if print_report:
        print(f"Rejected {np.sum(low_confidence_mask)} low-confidence samples as Trash.")

    y_true = np.argmax(generator.labels, axis=1)
    is_trash = np.sum(generator.labels, axis=1) == 0
    y_true[is_trash] = TRASH_INDEX
    
    report_classes = class_names + ["trash"]
    
    if print_report:
        print(classification_report(y_true, pred_classes, target_names=report_classes, zero_division=0))
    
    return classification_report(y_true, pred_classes, target_names=report_classes, output_dict=True, zero_division=0)

def get_optimal_treshold():
    best_treshold = 0.05
    best_f1 = 0

    for thresh in np.arange(0.1, 0.95, 0.05):
        report_valid_thresh = predict_with_threshold(valid_gen, threshold=thresh, print_report=False)
        report_test_thresh = predict_with_threshold(test_gen, threshold=thresh, print_report=False)

        f1 = np.mean([
            extract_average_f1(report_test_thresh), 
            extract_average_f1(report_valid_thresh)
        ])

        if(f1 > best_f1):
            best_treshold = thresh
            best_f1 = f1

    return best_treshold

THRESHOLD = get_optimal_treshold()

print("\n--- TEST DATA WITH THRESHOLD ---")
report_test_thresh = predict_with_threshold(test_gen, threshold=THRESHOLD)

print("\n--- VALIDATION DATA WITH THRESHOLD ---")
report_valid_thresh = predict_with_threshold(valid_gen, threshold=THRESHOLD)

avg_f1 = np.mean([
    extract_average_f1(report_test_thresh), 
    extract_average_f1(report_valid_thresh)
])
print(f"\nAverage f1 across test/valid: {avg_f1:.4f}")

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 150ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 150ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 147ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 142ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 141ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 141ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 138ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 138ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 152ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 152ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 167ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 